In [25]:
!pip install geopy


In [26]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import json

pd.set_option('display.max_columns', None)



In this step, I am importing the libraries that I need for the data work.  

* pandas – to work with tabular data  
* geopandas – to handle geographic data  
* osmnx– to retrieve pet store locations from OpenStreetMap  
* json– because some OSM data can come in JSON format  

I also used pd.set_option('display.max_columns', None) so that Jupyter shows all columns fully, without hiding or shortening them.



In [27]:
from geopy.geocoders import Nominatim
import time


In [28]:
tags = {"shop": "pet"}
petstores_gdf = ox.features_from_place("Berlin, Germany", tags=tags)
petstores_gdf.head()


geometry           brand brand:wikidata  \
element id                                                                    
node    111810614  POINT (13.28841 52.49974)       Fressnapf        Q875796   
        346138343  POINT (13.36752 52.54252)       Fressnapf        Q875796   
        417360251  POINT (13.45196 52.53361)       Fressnapf        Q875796   
        438385039  POINT (13.30844 52.47888)             NaN            NaN   
        446899194  POINT (13.44802 52.46867)  Das Futterhaus       Q1167914   

                     brand:wikipedia  check_date check_date:opening_hours  \
element id                                                                  
node    111810614       en:Fressnapf  2025-10-29               2025-10-29   
        346138343       de:Fressnapf  2025-08-28               2025-08-28   
        417360251       en:Fressnapf         NaN                      NaN   
        438385039                NaN         NaN                      NaN   
        446899194  de:Das Futterhaus         NaN                      NaN   

                             name                      opening_hours shop  \
element id                                                                  
node    111810614       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-19:00  pet   
        346138343       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-18:00  pet   
        417360251       Fressnapf                                NaN  pet   
        438385039  Lucas Tierwelt                  Mo-Sa 08:00-20:00  pet   
        446899194  Das Futterhaus                                NaN  pet   

                  addr:housenumber        addr:street level wheelchair  \
element id                                                               
node    111810614              NaN                NaN   NaN        NaN   
        346138343            10-11       Müllerstraße     0        yes   
        417360251             128b   Storkower Straße   NaN    limited   
        438385039               95  Forckenbeckstraße   NaN        NaN   
        446899194              NaN                NaN   NaN        NaN   

                  addr:city addr:country addr:postcode      addr:suburb  \
element id                                                                
node    111810614       NaN          NaN           NaN              NaN   
        346138343       NaN          NaN           NaN              NaN   
        417360251    Berlin           DE         10407  Prenzlauer Berg   
        438385039    Berlin           DE         14199    Schmargendorf   
        446899194       NaN          NaN           NaN              NaN   

                  toilets:wheelchair  \
element id                             
node    111810614                NaN   
        346138343                NaN   
        417360251                 no   
        438385039                NaN   
        446899194                NaN   

                                              wheelchair:description  \
element id                                                             
node    111810614                                                NaN   
        346138343                                                NaN   
        417360251  Behindertenparkplätze werden dauerhaft als Abs...   
        438385039                                                NaN   
        446899194                                                NaN   

                                          website  pet phone email  \
element id                                                           
node    111810614                             NaN  NaN   NaN   NaN   
        346138343                             NaN  NaN   NaN   NaN   
        417360251                             NaN  NaN   NaN   NaN   
        438385039  https://www.lucas-tierwelt.de/  NaN   NaN   NaN   
        446899194                             NaN  NaN   NaN   NaN   

                  payment:american_express payment:coins payment:mastercard  \
e

Here, I am retrieving pet shop locations in Berlin from OpenStreetMap.

* tags = {"shop": "pet"} selects only pet store locations  
* ox.features_from_place("Berlin, Germany", tags=tags) gets pet shops located in Berlin
* "Berlin, Germany" the place boundary used for the OSM query. 
* petstores_gdf.head() displays the first few rows to verify the data loaded correctly


In [29]:
neighborhoods = gpd.read_file("../../districts/sources/neighborhoods_enhanced.geojson")

print(neighborhoods.columns)

if "index_right" in petstores_gdf.columns:
    petstores_gdf = petstores_gdf.drop(columns=["index_right"])

petstores_gdf = gpd.sjoin(
    petstores_gdf,
    neighborhoods[["district", "neighborhood", "geometry"]],
    how="left",
    predicate="within",
    lsuffix="left",
    rsuffix="right"
)

petstores_gdf = petstores_gdf.rename(columns={
    "district_right": "district",
    "neighborhood_right": "neighborhood"
})

if "district_left" in petstores_gdf.columns:
    petstores_gdf = petstores_gdf.drop(columns=["district_left"])
if "neighborhood_left" in petstores_gdf.columns:
    petstores_gdf = petstores_gdf.drop(columns=["neighborhood_left"])

district_mapping = {
    'Mitte': '11001001',
    'Friedrichshain-Kreuzberg': '11002002',
    'Pankow': '11003003',
    'Charlottenburg-Wilmersdorf': '11004004',
    'Spandau': '11005005',
    'Steglitz-Zehlendorf': '11006006',
    'Tempelhof-Schöneberg': '11007007',
    'Neukölln': '11008008',
    'Treptow-Köpenick': '11009009',
    'Marzahn-Hellersdorf': '11010010',
    'Lichtenberg': '11011011',
    'Reinickendorf': '11012012'
}

petstores_gdf["district_id"] = petstores_gdf["district"].map(district_mapping)


Index(['district_id', 'district', 'neighborhood', 'geometry'], dtype='object')


In this step, I implemented the spatial join and applied the official district mapping to ensure consistency across the dataset.

What was done:

1-Loaded the neighborhoods layer (neighborhoods_enhanced.geojson)
This file contains the polygons for each neighborhood in Berlin and the corresponding district names.

2-Cleaned old spatial join artifacts
Removed index_right and potential leftover duplicate columns to avoid errors and ensure a clean join.

3-Performed a spatial join using predicate="within"
The pet store points were matched with the neighborhood polygons to determine:

-the correct district (name)
-the correct neighborhood (name)
-Only the necessary columns (district, neighborhood, geometry) were used to avoid inconsistent ID formats.

4-Renamed suffix columns
Reformatted district_right -> district and neighborhood_right -> neighborhood
This keeps the table clean and readable.

5-Applied the official district_id mapping
Using the dictionary provided in the mapping folder, each district name was mapped to the official Berlin district ID.This ensures full consistency with the project structure and other layers.

Result:
The dataset now contains:
-accurate spatial neighborhood assignments
-correct district names
-and standardized district_id values matching the official project mapping Fully aligned with the expected Layered project structure.

In [30]:
neighborhoods.head()


,district_id,district,neighborhood,geometry
0,01,Mitte,Mitte,"POLYGON ((13.41649 52.52696, 13.41635 52.52702..."
1,01,Mitte,Moabit,"POLYGON ((13.33884 52.51974, 13.33884 52.51974..."
2,01,Mitte,Hansaviertel,"POLYGON ((13.34322 52.51557, 13.34323 52.51557..."
3,01,Mitte,Tiergarten,"POLYGON ((13.36879 52.49878, 13.36891 52.49877..."
4,01,Mitte,Wedding,"POLYGON ((13.34656 52.53879, 13.34664 52.53878..."


In [31]:
print("Current CRS:", petstores_gdf.crs)


Current CRS: epsg:4326


In [32]:
print("Current CRS:", petstores_gdf.crs)


if petstores_gdf.crs != "EPSG:4326":
    petstores_gdf = petstores_gdf.to_crs(epsg=4326)
    print("CRS converted to EPSG:4326 (WGS 84)")

invalid_geometries = petstores_gdf[~petstores_gdf.is_valid]
print(f"Invalid geometries found: {len(invalid_geometries)}")

petstores_gdf = petstores_gdf.drop_duplicates(subset='geometry')

print("Geospatial validation completed successfully.")


Current CRS: epsg:4326
Invalid geometries found: 0
Geospatial validation completed successfully.


Validate and Clean Geospatial Data
* I confirmed that all geometries are valid and the coordinate reference system is EPSG:4326 (WGS 84), which is the standard for Berlin data.
No invalid geometries or duplicates were found, ensuring that the dataset is spatially accurate and ready for further cleaning and transformation.

In [33]:
petstores_gdf.columns.tolist()


['geometry',
 'brand',
 'brand:wikidata',
 'brand:wikipedia',
 'check_date',
 'check_date:opening_hours',
 'name',
 'opening_hours',
 'shop',
 'addr:housenumber',
 'addr:street',
 'level',
 'wheelchair',
 'addr:city',
 'addr:country',
 'addr:postcode',
 'addr:suburb',
 'toilets:wheelchair',
 'wheelchair:description',
 'website',
 'pet',
 'phone',
 'email',
 'payment:american_express',
 'payment:coins',
 'payment:mastercard',
 'payment:notes',
 'payment:visa',
 'contact:email',
 'contact:fax',
 'contact:phone',
 'contact:website',
 'note',
 'operator',
 'fax',
 'payment:credit_cards',
 'payment:debit_cards',
 'post_office',
 'post_office:brand',
 'post_office:brand:wikidata',
 'payment:girocard',
 'source',
 'contact:facebook',
 'delivery',
 'description',
 'species:de',
 'addr:inclusion',
 'contact:instagram',
 'ref:vatin',
 'dispensing',
 'opening_hours:covid19',
 'old_name',
 'wikimedia_commons',
 'surveillance',
 'start_date',
 'opening_hours:signed',
 'organic',
 'post_office:ref',

This lists all column names in the dataset to check which attributes are available and what information each represents.


In [34]:
petstores_gdf.isna().sum().sort_values(ascending=False)


entrance             95
wikimedia_commons    95
payment:girocard     95
contact:fax          95
delivery             95
                     ..
index_right           0
geometry              0
shop                  0
name                  0
district_id           0
Length: 74, dtype: int64

Here I am checking how many missing values each column has.

* .isna() marks missing entries
* .sum() counts how many missing values exist  
* .sort_values() sorts them from most to least  

This helps me understand data quality and decide which columns are useful.


In [35]:
petstores_gdf = petstores_gdf[petstores_gdf.geometry.type == "Point"].copy()


In [36]:
petstores_gdf["longitude"] = petstores_gdf.geometry.x
petstores_gdf["latitude"] = petstores_gdf.geometry.y


In [37]:
geolocator = Nominatim(user_agent="petstore_address_restoration")

def safe_reverse(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return None
    try:
        location = geolocator.reverse(f"{lat}, {lon}", timeout=10)
        time.sleep(1)
        return location.address if location else None
    except:
        return None



In [38]:
petstores_gdf["full_address"] = petstores_gdf.apply(
    lambda row: safe_reverse(row["latitude"], row["longitude"]),
    axis=1
)


In [39]:
petstores_gdf['full_address'].isna().sum()


np.int64(0)

In [40]:
keep_cols = [
    'geometry',
    'district_id',
    'name',
    'brand',
    'opening_hours',
    'addr:street',
    'addr:housenumber',
    'addr:postcode',
    'addr:city',
    'phone',
    'website',
    'full_address'
]
petstores_cleaned = petstores_gdf[keep_cols].copy()
petstores_cleaned.head()



geometry district_id            name  \
element id                                                                 
node    111810614  POINT (13.28841 52.49974)    11004004       Fressnapf   
        346138343  POINT (13.36752 52.54252)    11001001       Fressnapf   
        417360251  POINT (13.45196 52.53361)    11003003       Fressnapf   
        438385039  POINT (13.30844 52.47888)    11004004  Lucas Tierwelt   
        446899194  POINT (13.44802 52.46867)    11008008  Das Futterhaus   

                            brand                      opening_hours  \
element id                                                             
node    111810614       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-19:00   
        346138343       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-18:00   
        417360251       Fressnapf                                NaN   
        438385039             NaN                  Mo-Sa 08:00-20:00   
        446899194  Das Futterhaus                                NaN   

                         addr:street addr:housenumber addr:postcode addr:city  \
element id                                                                      
node    111810614                NaN              NaN           NaN       NaN   
        346138343       Müllerstraße            10-11           NaN       NaN   
        417360251   Storkower Straße             128b         10407    Berlin   
        438385039  Forckenbeckstraße               95         14199    Berlin   
        446899194                NaN              NaN           NaN       NaN   

                  phone                         website  \
element id                                                
node    111810614   NaN                             NaN   
        346138343   NaN                             NaN   
        417360251   NaN                             NaN   
        438385039   NaN  https://www.lucas-tierwelt.de/   
        446899194   NaN                             NaN   

                                                        full_address  
element id                                                            
node    111810614  Fressnapf, Lützenstraße, Halensee, Charlottenb...  
        346138343  Fressnapf, 10-11, Müllerstraße, Sprengelkiez, ...  
        417360251  Fressnapf, 128b, Storkower Straße, Blumenviert...  
        438385039  Lucas Tierwelt, 95, Forckenbeckstraße, Schmarg...  
        446899194  Das Futterhaus, 52, Lahnstraße, Richardkiez, N...

In this step, I manually selected the columns that are actually useful for the project.  
Before choosing, I checked all columns and their missing value counts.  
Some columns were almost completely empty
so they did not add meaningful value to the dataset.  






In [41]:
petstores_cleaned['geometry'] = petstores_cleaned['geometry'].centroid
petstores_cleaned['longitude'] = petstores_cleaned.geometry.x
petstores_cleaned['latitude'] = petstores_cleaned.geometry.y

petstores_cleaned[['name', 'full_address', 'latitude', 'longitude']].head()


/var/folders/mn/w43_rvg92n9dv58q76z80hgc0000gn/T/ipykernel_1990/2468793336.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  petstores_cleaned['geometry'] = petstores_cleaned['geometry'].centroid


name  \
element id                          
node    111810614       Fressnapf   
        346138343       Fressnapf   
        417360251       Fressnapf   
        438385039  Lucas Tierwelt   
        446899194  Das Futterhaus   

                                                        full_address  \
element id                                                             
node    111810614  Fressnapf, Lützenstraße, Halensee, Charlottenb...   
        346138343  Fressnapf, 10-11, Müllerstraße, Sprengelkiez, ...   
        417360251  Fressnapf, 128b, Storkower Straße, Blumenviert...   
        438385039  Lucas Tierwelt, 95, Forckenbeckstraße, Schmarg...   
        446899194  Das Futterhaus, 52, Lahnstraße, Richardkiez, N...   

                    latitude  longitude  
element id                               
node    111810614  52.499735  13.288411  
        346138343  52.542519  13.367520  
        417360251  52.533614  13.451963  
        438385039  52.478876  13.308440  
        446899194  52.468671  13.448016

The geometry field in OSM can sometimes be stored as shapes (polygons).  
However, for mapping and database storage we need a single coordinate point.  

* geometry.centroid extracts the center point of the shape.  
* geometry.x gives the longitude.  
* geometry.y gives the latitude.  

Then I display a preview to verify that the final coordinates were extracted correctly.



In [42]:
petstores_cleaned = petstores_cleaned.drop(columns=[
    'addr:street', 'addr:housenumber', 'addr:postcode', 'addr:city'
], errors='ignore')

petstores_cleaned.head()



geometry district_id            name  \
element id                                                                 
node    111810614  POINT (13.28841 52.49974)    11004004       Fressnapf   
        346138343  POINT (13.36752 52.54252)    11001001       Fressnapf   
        417360251  POINT (13.45196 52.53361)    11003003       Fressnapf   
        438385039  POINT (13.30844 52.47888)    11004004  Lucas Tierwelt   
        446899194  POINT (13.44802 52.46867)    11008008  Das Futterhaus   

                            brand                      opening_hours phone  \
element id                                                                   
node    111810614       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-19:00   NaN   
        346138343       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-18:00   NaN   
        417360251       Fressnapf                                NaN   NaN   
        438385039             NaN                  Mo-Sa 08:00-20:00   NaN   
        446899194  Das Futterhaus                                NaN   NaN   

                                          website  \
element id                                          
node    111810614                             NaN   
        346138343                             NaN   
        417360251                             NaN   
        438385039  https://www.lucas-tierwelt.de/   
        446899194                             NaN   

                                                        full_address  \
element id                                                             
node    111810614  Fressnapf, Lützenstraße, Halensee, Charlottenb...   
        346138343  Fressnapf, 10-11, Müllerstraße, Sprengelkiez, ...   
        417360251  Fressnapf, 128b, Storkower Straße, Blumenviert...   
        438385039  Lucas Tierwelt, 95, Forckenbeckstraße, Schmarg...   
        446899194  Das Futterhaus, 52, Lahnstraße, Richardkiez, N...   

                   longitude   latitude  
element id                               
node    111810614  13.288411  52.499735  
        346138343  13.367520  52.542519  
        417360251  13.451963  52.533614  
        438385039  13.308440  52.478876  
        446899194  13.448016  52.468671

Since the full address has already been combined into the full_address column,  
the original address fields (addr:street, addr:housenumber, addr:postcode, addr:city) are no longer needed.  

Therefore, I remove them using drop().  
The parameter errors="ignore" ensures that the code does not fail in case any of these columns are missing.


In [43]:
petstores_cleaned.info()


<class 'geopandas.geodataframe.GeoDataFrame'>
MultiIndex: 82 entries, ('node', np.int64(111810614)) to ('node', np.int64(13064860585))
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   geometry       82 non-null     geometry
 1   district_id    82 non-null     object  
 2   name           82 non-null     object  
 3   brand          38 non-null     object  
 4   opening_hours  65 non-null     object  
 5   phone          17 non-null     object  
 6   website        24 non-null     object  
 7   full_address   82 non-null     object  
 8   longitude      82 non-null     float64 
 9   latitude       82 non-null     float64 
dtypes: float64(2), geometry(1), object(7)
memory usage: 9.6+ KB


This allows me to verify:
* Number of rows and columns  
* Which columns still contain missing values  
* Data types  

It’s a quick validation step to make sure the cleaning worked properly.


In [44]:
petstores_cleaned.head(20)


geometry district_id  \
element id                                                  
node    111810614   POINT (13.28841 52.49974)    11004004   
        346138343   POINT (13.36752 52.54252)    11001001   
        417360251   POINT (13.45196 52.53361)    11003003   
        438385039   POINT (13.30844 52.47888)    11004004   
        446899194   POINT (13.44802 52.46867)    11008008   
        701825313   POINT (13.32671 52.49079)    11004004   
        800755193     POINT (13.3535 52.5746)    11012012   
        861870108   POINT (13.61083 52.54018)    11010010   
        1112653257  POINT (13.24238 52.42322)    11006006   
        1157588586   POINT (13.3666 52.56424)    11012012   
        1198087358  POINT (13.19667 52.53016)    11005005   
        1311479655  POINT (13.30647 52.52553)    11004004   
        1427998890  POINT (13.12883 52.52843)    11005005   
        1457600608  POINT (13.37072 52.56473)    11012012   
        1498303230  POINT (13.55979 52.50635)    11010010   
        1511328799   POINT (13.4244 52.48789)    11002002   
        1608667951  POINT (13.38438 52.50312)    11002002   
        1643105094  POINT (13.33338 52.59533)    11012012   
        1774557661  POINT (13.34222 52.46893)    11007007   
        1779217862  POINT (13.50724 52.49925)    11011011   

                                      name           brand  \
element id                                                   
node    111810614                Fressnapf       Fressnapf   
        346138343                Fressnapf       Fressnapf   
        417360251                Fressnapf       Fressnapf   
        438385039           Lucas Tierwelt             NaN   
        446899194           Das Futterhaus  Das Futterhaus   
        701825313           Natürlich Hund             NaN   
        800755193               Hundesalon             NaN   
        861870108                Fressnapf       Fressnapf   
        1112653257               Fressnapf       Fressnapf   
        1157588586          Das Futterhaus  Das Futterhaus   
        1198087358               Fressnapf       Fressnapf   
        1311479655            Zoo fridolin             NaN   
        1427998890               Fressnapf       Fressnapf   
        1457600608               Fressnapf       Fressnapf   
        1498303230          Das Futterhaus  Das Futterhaus   
        1511328799          Das Futterhaus  Das Futterhaus   
        1608667951               Fressnapf       Fressnapf   
        1643105094  Marine Aquarium Senior             NaN   
        1774557661      Steffis Futterkeks             NaN   
        1779217862               Fressnapf       Fressnapf   

                                                        opening_hours  \
element id                                                              
node    111810614                   Mo-Fr 09:00-20:00; Sa 09:00-19:00   
        346138343                   Mo-Fr 09:00-20:00; Sa 09:00-18:00   
        417360251                                                 NaN   
        438385039                                   Mo-Sa 08:00-20:00   
        446899194                                                 NaN   
        701825313                   Tu-Fr 11:00-18:00; Sa 10:00-15:00   
        800755193   "nach Vereinbarung (Beschriftung der Öffnungsz...   
        861870108           Mo-Fr 09:00-20:00; Sa 09:00-18:00; PH off   
        1112653257                                                NaN   
        1157588586                                             closed   
        1198087358                       Mo-Sa 09:00-20:00; Su,PH off   
        1311479655  Mo-Fr 09:00-18:00; Sa 09:00-14:00; Su off, PH off   
        1427998890                                                NaN   
        1457600608                  Mo-Fr 09:00-20:00; Sa 09:00-18:00   
        1498303230                                                NaN   
        1511328799                                  Mo-Sa 10:00-20:00   
        1608667

Using petstores_cleaned.head(20) I preview the first 20 rows of the cleaned dataset.  

This helps me visually confirm that:  
* The selected columns are correct  
* The full_address column was created properly  
* Latitude and longitude values look valid  
* The data structure matches the expected format


In [45]:
print("Final dataset shape:", petstores_cleaned.shape)
print("Columns:", list(petstores_cleaned.columns))
print("Any missing values left?", petstores_cleaned.isna().any().sum())


Final dataset shape: (82, 10)
Columns: ['geometry', 'district_id', 'name', 'brand', 'opening_hours', 'phone', 'website', 'full_address', 'longitude', 'latitude']
Any missing values left? 4


In [46]:
petstores_cleaned.isna().sum()


geometry          0
district_id       0
name              0
brand            44
opening_hours    17
phone            65
website          58
full_address      0
longitude         0
latitude          0
dtype: int64

In [47]:
petstores_gdf.head()


geometry           brand brand:wikidata  \
element id                                                                    
node    111810614  POINT (13.28841 52.49974)       Fressnapf        Q875796   
        346138343  POINT (13.36752 52.54252)       Fressnapf        Q875796   
        417360251  POINT (13.45196 52.53361)       Fressnapf        Q875796   
        438385039  POINT (13.30844 52.47888)             NaN            NaN   
        446899194  POINT (13.44802 52.46867)  Das Futterhaus       Q1167914   

                     brand:wikipedia  check_date check_date:opening_hours  \
element id                                                                  
node    111810614       en:Fressnapf  2025-10-29               2025-10-29   
        346138343       de:Fressnapf  2025-08-28               2025-08-28   
        417360251       en:Fressnapf         NaN                      NaN   
        438385039                NaN         NaN                      NaN   
        446899194  de:Das Futterhaus         NaN                      NaN   

                             name                      opening_hours shop  \
element id                                                                  
node    111810614       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-19:00  pet   
        346138343       Fressnapf  Mo-Fr 09:00-20:00; Sa 09:00-18:00  pet   
        417360251       Fressnapf                                NaN  pet   
        438385039  Lucas Tierwelt                  Mo-Sa 08:00-20:00  pet   
        446899194  Das Futterhaus                                NaN  pet   

                  addr:housenumber        addr:street level wheelchair  \
element id                                                               
node    111810614              NaN                NaN   NaN        NaN   
        346138343            10-11       Müllerstraße     0        yes   
        417360251             128b   Storkower Straße   NaN    limited   
        438385039               95  Forckenbeckstraße   NaN        NaN   
        446899194              NaN                NaN   NaN        NaN   

                  addr:city addr:country addr:postcode      addr:suburb  \
element id                                                                
node    111810614       NaN          NaN           NaN              NaN   
        346138343       NaN          NaN           NaN              NaN   
        417360251    Berlin           DE         10407  Prenzlauer Berg   
        438385039    Berlin           DE         14199    Schmargendorf   
        446899194       NaN          NaN           NaN              NaN   

                  toilets:wheelchair  \
element id                             
node    111810614                NaN   
        346138343                NaN   
        417360251                 no   
        438385039                NaN   
        446899194                NaN   

                                              wheelchair:description  \
element id                                                             
node    111810614                                                NaN   
        346138343                                                NaN   
        417360251  Behindertenparkplätze werden dauerhaft als Abs...   
        438385039                                                NaN   
        446899194                                                NaN   

                                          website  pet phone email  \
element id                                                           
node    111810614                             NaN  NaN   NaN   NaN   
        346138343                             NaN  NaN   NaN   NaN   
        417360251                             NaN  NaN   NaN   NaN   
        438385039  https://www.lucas-tierwelt.de/  NaN   NaN   NaN   
        446899194                             NaN  NaN   NaN   NaN   

                  payment:american_express payment:coins payment:mastercard  \
e

In [48]:
petstores_cleaned.to_csv("berlin_pet_stores_cleaned.csv", index=False)


* This line allows me to export the cleaned dataset. So I am saving the petstores_cleanedtable as a CSV file on my computer .

* berlin_pet_stores_cleaned.csv -- This is the name of the file I am saving.  
* index=False → I am not including row index numbers in the CSV because they are not needed. 

Data Cleaning & Standardization Notes

Steps I Followed During Data Cleaning

* I first loaded the raw OSM data into a GeoDataFrame called petstores_gdf.

* I reviewed the full list of available columns to understand what information the dataset contained.

* I used isna().sum() to check the amount of missing data in each column.

* Columns that were mostly empty or not relevant to the project were removed.

* I selected a meaningful subset of columns and created a cleaned dataset: petstores_cleaned.

Standardizing Columns

* Address fields (addr:street, addr:housenumber, addr:postcode, addr:city) were combined into a single, readable full_address column.

* To work with geographic data more easily, I extracted latitude and longitude from the geometry column.

* The original, now redundant address columns were removed.


Data Sources and Integration

* At this stage, only OSM was used as the primary data source.

 Challenges I Faced

* The most challenging part for me was deciding which columns were actually useful.
The raw OSM dataset contains many fields, and the column names are not always intuitive.
So I needed time to understand what each field represented.

* Using isna().sum() to examine missing data helped me see which columns had meaningful values and which were mostly empty.
This made the column selection process much clearer.

* I also needed to pay attention when creating the full_address column because some rows were missing one or more address components.
To handle this, I used fillna(''), which allowed me to merge the address parts cleanly without errors.

* Overall, once I understood the structure of the data, the workflow became more natural and easier to follow.

Deciding Which Data to Keep

 I kept these columns because they provide direct value to the user and to the application:

* name — Important for identifying the store

* geometry — Contains the raw geographic location data

* brand — Needed for segmenting and categorizing stores by brand

* full_address — Provides a clean and readable address format

* opening_hours — Useful for user decision-making

* phone, website — Helps with direct communication

* latitude, longitude — Necessary for map display and spatial analysis

 Columns removed — because they were too incomplete or not relevant:

* Payment-related fields (payment:*)

* OSM metadata (source, wikidata)